In [1]:
!pip install transformers torch tqdm

In [8]:
import json
import torch
from tqdm import tqdm
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
import os

In [3]:
# 1. Load Model

model_name = "facebook/bart-large-cnn"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

device = "cuda" if torch.cuda.is_available() else "cpu"
model = model.to(device)


# 2. Load Raw Dataset


with open("Original_dataset.json", "r", encoding="utf-8") as f:
    raw_data = json.load(f)





/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/1.63G [00:00<?, ?B/s]

Please make sure the generation config includes `forced_bos_token_id=0`. 


Loading weights:   0%|          | 0/511 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/363 [00:00<?, ?B/s]

In [4]:

# 3. Prompt Engineering


def generate_text(prompt, max_length=512):
    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=1024
    ).to(device)

    summary_ids = model.generate(
        inputs["input_ids"],
        max_length=max_length,
        num_beams=4,
        early_stopping=True
    )

    return tokenizer.decode(summary_ids[0], skip_special_tokens=True)



In [5]:

# 4. Structured Processing Function


def process_article(article):

    original_text = article["Original_Description"]

    # -------- Simplified Explanation Prompt --------
    simplify_prompt = f"""
    Simplify the following Constitutional Article in easy student-friendly language.
    Explain clearly in paragraphs and add one practical example starting with 'Ex :'.

    Article Text:
    {original_text}
    """

    simplified = generate_text(simplify_prompt, max_length=400)

    # -------- Key Points Prompt --------
    keypoints_prompt = f"""
    Extract 4-5 important key points from the following constitutional article.
    Write them in bullet format.

    {original_text}
    """

    key_points = generate_text(keypoints_prompt, max_length=200)

    # -------- Historical Context Prompt --------
    history_prompt = f"""
    Provide brief historical context about the following constitutional article.
    Keep it concise and informative.

    {original_text}
    """

    historical_context = generate_text(history_prompt, max_length=200)

    # -------- Landmark Cases Prompt --------
    case_prompt = f"""
    Mention important Supreme Court landmark cases related to the following constitutional article.
    If none are directly related, say 'No major landmark case directly associated.'

    {original_text}
    """

    landmark_cases = generate_text(case_prompt, max_length=200)

    # -------- Related Articles Prompt --------
    related_prompt = f"""
    Suggest related Articles of the Indian Constitution connected to the following article.
    Provide in short list format.

    {original_text}
    """

    related_articles = generate_text(related_prompt, max_length=150)

    # -------- Part Classification (Simple Rule-Based Logic) --------
    article_number = article["Article"]

    if article_number == "0":
        part = "Preamble"
        subject = "Preamble"
    elif 1 <= int(article_number) <= 4:
        part = "Part I"
        subject = "Union & Its Territory"
    elif 5 <= int(article_number) <= 11:
        part = "Part II"
        subject = "Citizenship"
    elif 12 <= int(article_number) <= 35:
        part = "Part III"
        subject = "Fundamental Rights"
    else:
        part = "Other"
        subject = "Other"

    # -------- Final Structured Output --------
    structured_output = {
        "Article": article["Article"],
        "Title": article["Title"],
        "Original_Description": original_text,
        "Simplified_Description": simplified,
        "Part": part,
        "Subject": subject,
        "Key_Points": key_points,
        "Historical_Context": historical_context,
        "Landmark_Cases": landmark_cases,
        "Related_Articles": related_articles
    }

    return structured_output



In [ ]:
# 5. Process Entire Dataset

output_file = "Prepared_dataset.json"

#Resume from existing processed data if available
if os.path.exists(output_file):
    with open(output_file, "r", encoding="utf-8") as f:
        prepared_dataset = json.load(f)
else:
    prepared_dataset = []
    print(" Starting fresh processing...")

# Get already processed article numbers
processed_articles = set(item["Article"] for item in prepared_dataset)


# 6. Process Articles


for article in tqdm(raw_data):

    if article["Article"] in processed_articles:
        continue  

    processed = process_article(article)
    prepared_dataset.append(processed)

    # Save checkpoint after every article
    with open(output_file, "w", encoding="utf-8") as f:
        json.dump(prepared_dataset, f, indent=4, ensure_ascii=False)

print(" Processing complete!")


 Starting fresh processing...


100%|██████████| 467/467 [00:00<00:00, 727074.97it/s]

 Processing complete!
